# 05 — Trading agents

**Phase 5 deliverable:** all 23 agents backtested on KBANK, frictionless and with frictions.

23 notebooks, 6 families. The Q-learning set alone is 11 notebooks sharing one replay-buffer
skeleton with a swapped head — implemented here as `{double, duel, recurrent, curiosity}` flags
that compose.

## Two corrections to upstream, both load-bearing

**Agents get a real holdout.** In `agent/6.evolution-strategy-agent.ipynb`, `get_reward()` (the
training objective) and `buy()` (the reported equity curve) iterate the *same* `self.trend`. Every
published agent return in that repository is in-sample. Here `fit` sees the training block and the
reported result comes from the test block that follows it.

**The `close[t]` global is fixed, not ported.** Notebook 6's buy branch reads
`starting_money -= close[t]` — a module-level global — where the sell branch correctly reads
`self.trend[t]`. It is silent only because the two happen to hold the same list, and it detonates
the first time two tickers share a process. Nothing here reads a global: cash and inventory live
on the environment during training and on `SETMarket` during evaluation.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

In [2]:
from stock_retrofit.config import all_agent_specs
from stock_retrofit.agents import registered_kinds

print("agent families:", ", ".join(registered_kinds()), "\n")
for spec in all_agent_specs():
    print(f"  {spec.name:36s} {spec.kind:20s} <- {spec.upstream}")

agent families: abcd, actor_critic, buy_and_hold, evolution_strategy, mean_reversion, moving_average, neuro_evolution, policy_gradient, q_learning, signal_rolling, turtle 

  00_buy_and_hold                      buy_and_hold         <- (baseline — not an upstream notebook)
  01_turtle                            turtle               <- agent/1.turtle-agent.ipynb
  02_moving_average                    moving_average       <- agent/2.moving-average-agent.ipynb
  03_signal_rolling                    signal_rolling       <- agent/3.signal-rolling-agent.ipynb
  04_policy_gradient                   policy_gradient      <- agent/4.policy-gradient-agent.ipynb
  05_q_learning                        q_learning           <- agent/5.q-learning-agent.ipynb
  06_evolution_strategy                evolution_strategy   <- agent/6.evolution-strategy-agent.ipynb
  07_double_q_learning                 q_learning           <- agent/7.double-q-learning-agent.ipynb
  08_recurrent_q_learning              q_lea

## How the 11 Q-learning notebooks become one class

| flag | what it does |
|---|---|
| `double` | the online net picks the next action, the target net scores it — removes the max operator's optimism bias |
| `duel` | the head splits into value and advantage streams recombined as `V + (A − mean A)` |
| `recurrent` | an LSTM reads the window as a sequence instead of an MLP reading it flattened |
| `curiosity` | a forward model's prediction error is added to the reward as an exploration bonus |

In [3]:
specs = {s.name: s for s in all_agent_specs()}
rows = []
for name, spec in specs.items():
    if spec.kind == "q_learning":
        p = spec.params
        rows.append({"config": name,
                     "double": p.get("double"), "duel": p.get("duel"),
                     "recurrent": p.get("recurrent"), "curiosity": p.get("curiosity"),
                     "upstream": spec.upstream.split("/")[-1]})
pd.DataFrame(rows)

,config,double,duel,recurrent,curiosity,upstream
0,05_q_learning,False,False,False,False,5.q-learning-agent.ipynb
1,07_double_q_learning,True,False,False,False,7.double-q-learning-agent.ipynb
2,08_recurrent_q_learning,False,False,True,False,8.recurrent-q-learning-agent.ipynb
3,09_double_recurrent_q_learning,True,False,True,False,9.double-recurrent-q-learning-agent.ipynb
4,10_duel_q_learning,False,True,False,False,10.duel-q-learning-agent.ipynb
5,11_double_duel_q_learning,True,True,False,False,11.double-duel-q-learning-agent.ipynb
6,12_duel_recurrent_q_learning,False,True,True,False,12.duel-recurrent-q-learning-agent.ipynb
7,13_double_duel_recurrent_q_learning,True,True,True,False,13.double-duel-recurrent-q-learning-agent.ipynb
8,18_curiosity_q_learning,False,False,False,True,18.curiosity-q-learning-agent.ipynb
9,19_recurrent_curiosity_q_learning,False,False,True,True,19.recurrent-curiosity-q-learning-agent.ipynb


## A note on training speed

Learning agents need tens of thousands of simulated steps per fold, so **training** runs against a
fast numpy environment with a proportional round-trip cost. **Evaluation never does** — every
reported number comes from replaying the trained policy through `SETMarket` with board lots, tick
snapping, commission + VAT, price limits and the participation cap all enforced.

Training may use any objective it likes. Only the evaluation is a claim.

## Run the whole catalogue

Equivalent to:

```bash
python -m stock_retrofit.cli backtest --all --symbol KBANK
```

This takes roughly 20 minutes on CPU — the recurrent Q-learning variants dominate.

In [4]:
from stock_retrofit.paths import RESULTS_DIR
from stock_retrofit.report import backtest_symbol

cached = RESULTS_DIR / "backtest-KBANK.csv"
agents = pd.read_csv(cached) if cached.exists() else backtest_symbol("KBANK")
agents

,agent,symbol,folds,ret_friction,ret_frictionless,friction_gap,sharpe_friction,sharpe_frictionless,trades,costs,max_dd,status
0,00_buy_and_hold,KBANK,8,0.078205,0.080884,0.002680,1.409520,1.445892,8,13313.207500,-0.113256,ok
1,04_policy_gradient,KBANK,8,0.057471,0.066294,0.008824,1.382111,1.579088,37,63535.833880,-0.164200,ok
2,13_double_duel_recurrent_q_learning,KBANK,8,0.054388,0.062658,0.008270,1.515287,1.726188,36,62295.227730,-0.049714,ok
3,09_double_recurrent_q_learning,KBANK,8,0.053270,0.064554,0.011284,1.593626,1.907851,49,83089.533900,-0.070026,ok
4,16_actor_critic_recurrent,KBANK,8,0.049756,0.051974,0.002218,1.468930,1.521264,10,15053.919880,-0.050636,ok
5,08_recurrent_q_learning,KBANK,8,0.044397,0.057289,0.012892,1.395236,1.769728,57,98227.112800,-0.058105,ok
6,19_recurrent_curiosity_q_learning,KBANK,8,0.043516,0.051487,0.007971,1.216456,1.412890,34,57069.142825,-0.060651,ok
7,21_neuro_evolution,KBANK,8,0.039334,0.053917,0.014583,1.047236,1.417742,59,100624.246105,-0.122608,ok
8,17_actor_critic_duel_recurrent,KBANK,8,0.039125,0.040294,0.001169,1.100607,1.123683,4,6651.900030,-0.107783,ok
9,07_double_q_learning,KBANK,8,0.037534,0.062757,0.025223,1.169864,1.912308,113,190681.837215,-0.052284,ok


## Reading the result

`ret_frictionless` vs `ret_friction` is the headline (spec R11). The gap is what it costs to stop
pretending the market has no board lots, no tick sizes, no commission and no VAT.

Buy-and-hold is pinned to the top as the agent baseline — the counterpart to `NaiveLag` on the
model tables. An agent that trades hard and lands below it has bought turnover, not alpha.

In [5]:
ok = agents[agents["status"] == "ok"]
print(f"profitable frictionless : {int((ok['ret_frictionless'] > 0).sum())} of {len(ok)}")
print(f"profitable after costs  : {int((ok['ret_friction'] > 0).sum())} of {len(ok)}")
print(f"mean cost of frictions  : {ok['friction_gap'].mean():+.2%} per fold")

baseline = ok.loc[ok["agent"].str.contains("buy_and_hold"), "ret_friction"].iloc[0]
beat = ok[(ok["ret_friction"] > baseline) & (~ok["agent"].str.contains("buy_and_hold"))]
print(f"\nbeat buy-and-hold after costs: {len(beat)} of {len(ok) - 1}")
ok.nlargest(8, "friction_gap")[["agent", "trades", "ret_frictionless", "ret_friction", "friction_gap"]]

profitable frictionless : 24 of 24
profitable after costs  : 20 of 24
mean cost of frictions  : +1.16% per fold

beat buy-and-hold after costs: 0 of 23


,agent,trades,ret_frictionless,ret_friction,friction_gap
21,10_duel_q_learning,126,0.022747,-0.003967,0.026713
23,05_q_learning,122,0.020124,-0.005281,0.025405
9,07_double_q_learning,113,0.062757,0.037534,0.025223
14,11_double_duel_q_learning,108,0.048950,0.025434,0.023516
11,03_signal_rolling,90,0.050713,0.031354,0.019360
13,22_neuro_evolution_novelty,77,0.047751,0.030867,0.016885
22,06_evolution_strategy,79,0.012275,-0.004343,0.016618
15,18_curiosity_q_learning,77,0.039805,0.023230,0.016576
